In [1]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


# ----------------------------
# 1) World model (Blocks World)
# ----------------------------

@dataclass(frozen=True)
class Obj:
    name: str
    color: str
    shape: str  # "cube", "pyramid", "block" (generic)

@dataclass
class World:
    objects: Dict[str, Obj]
    whoIsUnder: Dict[str, Optional[str]]  # on[x] = y means x is on y; None means on table
    #If you have on["block_a"] = "block_b", it means block_a is physically sitting on top of block_b
    holding: Optional[str] = None #The agent is currently holding the object associated with that name.

    def is_clear(self, obj_name: str) -> bool:
        """True if nothing is on top of obj_name."""
        # return all(support != obj_name for support in self.on.values())
        for support in self.whoIsUnder.values():
            if support == obj_name:
                return False
        return True

    def top_of(self, obj_name: str) -> Optional[str]:
        """Return object that is on top of obj_name (if any)."""
        for x, support in self.whoIsUnder.items():
            print(x,support)
            if support == obj_name:
                return x
        return None

    def describe(self) -> str:
        #describes all relationships between objects and if world agent is holding something
        lines = []
        for x in sorted(self.objects):
            support = self.whoIsUnder.get(x, None)
            if support is None:
                lines.append(f"{x} is on the table")
            else:
                lines.append(f"{x} is on {support}")
        lines.append(f"Holding: {self.holding}")
        return "\n".join(lines)

    # ----------------------------
    # 4) Execution (actions)
    # ----------------------------
    def pickup(self, x: str) -> None:
        #pick up obj if already not holding anything and if object isnt under something else
        if self.holding is not None:
            raise RuntimeError("Already holding something.")
        if not self.is_clear(x):
            raise RuntimeError(f"Cannot pick up {x}: not clear.")
        self.holding = x
        self.whoIsUnder[x] = None

    def put_on(self, x: str, y: str) -> None:
        #put down obj on another obj, if holding something and if target object isnt under something else / blocked
        if self.holding != x:
            raise RuntimeError(f"Not holding {x}.")
        if not self.is_clear(y):
            raise RuntimeError(f"Cannot place on {y}: {y} not clear.")
        self.whoIsUnder[x] = y
        self.holding = None



In [2]:
world1 = World(
    objects={
        "bottom": Obj("bottom", "red", "cube"),
        "b2": Obj("b2", "red", "cube"),     # deliberately ambiguous for "red cube"
        "top": Obj("top", "blue", "pyramid"),
        "c1": Obj("c1", "green", "cube"),
    },
    whoIsUnder={
        "top": "bottom",   # blue pyramid on red cube (bottom)
        "bottom": None,
        "b2": None,
        "c1": None,
    },
)
blocker1 = world1.top_of("bottom")

world1.describe()

top bottom


'b2 is on the table\nbottom is on the table\nc1 is on the table\ntop is on bottom\nHolding: None'

In [3]:

# ----------------------------------------
# 2) Reference grounding (simple semantics)
# ----------------------------------------

def resolve_ref(world: World, color: Optional[str], shape: Optional[str]) -> List[str]:
    """Return object names matching the requested properties."""
    matches = []
    for name, obj in world.objects.items():
        if color is not None and obj.color != color:
            continue
        if shape is not None:
            # allow 'block' as a generic term (matches anything)
            if shape != "block" and obj.shape != shape:
                continue
        matches.append(name)
    return matches



In [4]:
resolve_ref(world1, "red", "cube")

['bottom', 'b2']

In [5]:

# ----------------------------------------
# 3) Planning / inference (toy planner)
# ----------------------------------------

def plan_pickup(world: World, x: str) -> List[Tuple[str, str, Optional[str]]]:
    """
    Plan to pick up x. If x is not clear, move blockers to table first.
    Returns a plan as a list of (action, obj, target).
    """
    plan = []
    blocker = world.top_of(x)
    if blocker is not None:
        # move blocker away first (to table)
        # if blocker itself isn't clear, recurse
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))  # put on table
    plan.append(("pickup", x, None))
    return plan


def plan_put(world: World, x: str, y: str) -> List[Tuple[str, str, Optional[str]]]:
    """Plan to put x on y, ensuring both are feasible."""
    plan = []
    # Ensure y is clear
    blocker = world.top_of(y)
    if blocker is not None:
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))
    # Ensure we can pick up x
    plan += plan_pickup(world, x)
    plan.append(("put_on", x, y))
    return plan


def execute_plan(world: World, plan: List[Tuple[str, str, Optional[str]]]) -> None:
    for action, obj, target in plan:
        if action == "pickup":
            world.pickup(obj)
        elif action == "put_on":
            if target is None:
                # put on table
                if world.holding != obj:
                    # if we planned it, we should be holding it; but safe-check
                    raise RuntimeError("Planner/executor mismatch.")
                world.whoIsUnder[obj] = None
                world.holding = None
            else:
                world.put_on(obj, target)
        else:
            raise ValueError(f"Unknown action: {action}")




In [6]:
# ----------------------------------------
# 1+2) "Parsing": minimal, rule-based parser
# ----------------------------------------

COLORS = {"red", "green", "blue", "yellow"}
SHAPES = {"cube", "pyramid", "block"}


#RULESET
def parse_command(text: str) -> dict:
    """
    Extremely small rule-based parser for demo:
    - "pick up the red block"
    - "put the blue pyramid on the red cube"
    """
    t = text.lower().replace("?", "").strip()
    tokens = t.split()

    if t.startswith("pick up"):
        # find color + shape (if present)
        color = next((w for w in tokens if w in COLORS), None)
        shape = next((w for w in tokens if w in SHAPES), "block")
        return {"intent": "PICKUP", "ref": {"color": color, "shape": shape}}

    if t.startswith("put"):
        # "put the X on the Y"
        if "on" not in tokens:
            raise ValueError("Expected 'on' in put command.")
        on_i = tokens.index("on")
        left = tokens[1:on_i]     # description of X
        right = tokens[on_i+1:]   # description of Y

        color_x = next((w for w in left if w in COLORS), None)
        shape_x = next((w for w in left if w in SHAPES), "block")
        color_y = next((w for w in right if w in COLORS), None)
        shape_y = next((w for w in right if w in SHAPES), "block")

        return {
            "intent": "PUT_ON",
            "x": {"color": color_x, "shape": shape_x},
            "y": {"color": color_y, "shape": shape_y},
        }

    raise ValueError("Unknown command format.")




In [7]:
# ----------------------------------------
# Dialogue manager: clarification behavior
# ----------------------------------------

def choose_unique(matches: List[str], what: str) -> str:
    if not matches:
        raise ValueError(f"I can't find any {what} that matches your description.")
    if len(matches) > 1:
        # SHRDLU-like clarification
        raise ValueError(f"I don't understand which {what} you mean: {matches}")
    return matches[0]


def interpret_and_act(world: World, utterance: str) -> None:
    parsed = parse_command(utterance)

    if parsed["intent"] == "PICKUP":
        ref = parsed["ref"]
        matches = resolve_ref(world, ref["color"], ref["shape"])
        x = choose_unique(matches, f"{ref['color'] or ''} {ref['shape']}".strip())
        plan = plan_pickup(world, x)
        print("PLAN:", plan)
        execute_plan(world, plan)
        print(f"OK. Picked up {x}.")

    elif parsed["intent"] == "PUT_ON":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"])
        my = resolve_ref(world, parsed["y"]["color"], parsed["y"]["shape"])
        x = choose_unique(mx, f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        y = choose_unique(my, f"{parsed['y']['color'] or ''} {parsed['y']['shape']}".strip())
        plan = plan_put(world, x, y)
        print("PLAN:", plan)
        execute_plan(world, plan)
        print(f"OK. Put {x} on {y}.")

    else:
        raise ValueError("Unsupported intent.")




In [8]:
# ----------------------------
# Demo run
# ----------------------------

if __name__ == "__main__":
    world = World(
        objects={
            "b1": Obj("b1", "red", "cube"),
            "b2": Obj("b2", "red", "cube"),     # deliberately ambiguous for "red cube"
            "p1": Obj("p1", "blue", "pyramid"),
            "c1": Obj("c1", "green", "cube"),
        },
        whoIsUnder={
            "p1": "b1",   # blue pyramid on red cube (b1)
            "b1": None,
            "b2": None,
            "c1": None,
        },
    )

    print("INITIAL WORLD:\n" + world.describe(), "\n")

    # 1) Example that triggers clarification (two red cubes exist)
    try:
        interpret_and_act(world, "put the blue pyramid on the red cube")
    except ValueError as e:
        print("SHRDLU:", e)

    # 2) Disambiguated request
    print("\nDisambiguating by specifying the target name isn't supported by our tiny grammar,")
    print("so we instead remove ambiguity by changing the world (like a controlled micro-domain).")

    # Remove one red cube to resolve ambiguity
    del world.objects["b2"]
    del world.whoIsUnder["b2"]

    print("\nWORLD NOW:\n" + world.describe(), "\n")

    interpret_and_act(world, "put the red cube on the blue pyramid")

    print("\nFINAL WORLD:\n" + world.describe())

INITIAL WORLD:
b1 is on the table
b2 is on the table
c1 is on the table
p1 is on b1
Holding: None 

SHRDLU: I don't understand which red cube you mean: ['b1', 'b2']

Disambiguating by specifying the target name isn't supported by our tiny grammar,
so we instead remove ambiguity by changing the world (like a controlled micro-domain).

WORLD NOW:
b1 is on the table
c1 is on the table
p1 is on b1
Holding: None 

p1 b1
b1 None
c1 None
p1 b1
p1 b1
b1 None
c1 None
PLAN: [('pickup', 'p1', None), ('put_on', 'p1', None), ('pickup', 'b1', None), ('put_on', 'b1', 'p1')]
OK. Put b1 on p1.

FINAL WORLD:
b1 is on p1
c1 is on the table
p1 is on the table
Holding: None
